# Paper 3 · Phase 1b — DroneDetect features

One pass over the raw recordings (external drive, ~22 min). Each recording is cut into non-overlapping 4000-sample windows,
and windows are selected by DC-removed power (`select_windows`):
- **active**: ≥ 6 dB above the recording's 10th-percentile floor;
- **noise-only**: ≤ the floor, kept for the receiver-shortcut test.

Up to 1000 of each are kept per recording, spread over time. Each window → DC removal, unit power, WPD(I) ‖ WPD(Q) = 256-d
(`iq_wpd_features`, the DroneRF WPD).

Per-recording results are cached in `results/paper3/phase1/features/{rec_id}.npz` (resume by skipping existing files), then
concatenated into `results/paper3/phase1/features.npz`.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from joblib import Parallel, delayed
from tqdm import tqdm

from rfml_uav.drone_detect.dataset import inventory, read, select_windows
from rfml_uav.data.wavelets import iq_wpd_features

load_dotenv()
out_path = os.path.join(os.getenv("DATA_PATH"), "results/paper3/phase1")
cache_path = os.path.join(out_path, "features")
os.makedirs(cache_path, exist_ok=True)

WINDOW, CAP = 4000, 1000
LIMIT = None  # set to a small int to process only the first recordings
inv = inventory(os.getenv("DRONEDETECT_PATH"))
print(len(inv), "recordings")

390 recordings


In [2]:
def extract(path, rec_id):
    warnings.filterwarnings("ignore")  # pywt: level 6 is deep for 4000 samples (same setting as DroneRF)
    out = os.path.join(cache_path, rec_id + ".npz")
    if os.path.exists(out):
        return
    x = read(path)
    active, noise, active_frac = select_windows(x, size=WINDOW, cap=CAP)
    starts = np.concatenate([active, noise])
    order = np.argsort(starts, kind="stable")
    starts, is_active = starts[order], np.r_[np.ones(len(active), bool), np.zeros(len(noise), bool)][order]
    X = np.stack([iq_wpd_features(x[s:s + WINDOW]) for s in starts])
    np.savez(out + ".tmp.npz", X=X, start=starts, active=is_active, active_frac=active_frac)
    os.replace(out + ".tmp.npz", out)  # atomic: an interrupted run never leaves a half-written file

todo = inv if LIMIT is None else inv.head(LIMIT)
todo = todo[[not os.path.exists(os.path.join(cache_path, r + ".npz")) for r in todo.rec_id]]
print(len(todo), "to extract")
_ = Parallel(n_jobs=8)(delayed(extract)(p, r) for p, r in tqdm(zip(todo.path, todo.rec_id), total=len(todo)))

388 to extract


  0%|          | 0/388 [00:00<?, ?it/s]

  4%|▍         | 16/388 [00:24<09:39,  1.56s/it]

  6%|▌         | 24/388 [00:33<08:11,  1.35s/it]

  8%|▊         | 32/388 [00:51<09:56,  1.68s/it]

...

 93%|█████████▎| 360/388 [16:19<01:09,  2.47s/it]

 95%|█████████▍| 368/388 [16:41<00:51,  2.56s/it]

 97%|█████████▋| 376/388 [17:04<00:32,  2.67s/it]

 99%|█████████▉| 384/388 [17:28<00:11,  2.76s/it]

100%|██████████| 388/388 [17:28<00:00,  2.70s/it]

## Concatenate

In [3]:
cols = {k: [] for k in ["X", "subset", "drone", "mode", "k", "rec_id", "start", "active"]}
fracs = []
for r in inv.itertuples():
    f = os.path.join(cache_path, r.rec_id + ".npz")
    if not os.path.exists(f):
        continue
    d = np.load(f)
    n = len(d["X"])
    cols["X"].append(d["X"]); cols["start"].append(d["start"]); cols["active"].append(d["active"])
    for c in ["subset", "drone", "mode", "k", "rec_id"]:
        cols[c].append(np.full(n, getattr(r, c)))
    fracs.append({"subset": r.subset, "drone": r.drone, "active_frac": float(d["active_frac"]),
                  "n_active": int(d["active"].sum()), "n_noise": int((~d["active"]).sum())})
F = {k: np.concatenate(v) for k, v in cols.items()}
assert not np.isnan(F["X"]).any()
assert all(np.all(np.diff(F["start"][F["rec_id"] == r]) > 0) for r in np.unique(F["rec_id"]))
print(len(np.unique(F["rec_id"])), "recordings,", F["X"].shape, "rows")
np.savez(os.path.join(out_path, "features.npz"), **F)

fr = pd.DataFrame(fracs)
fr.pivot_table(index="drone", columns="subset", values=["active_frac", "n_active", "n_noise"], aggfunc={"active_frac": "mean", "n_active": "sum", "n_noise": "sum"}).round(3)

390 recordings, (780000, 256) rows


active_frac                      n_active                      n_noise  \
subset        BLUE   BOTH  CLEAN   WIFI     BLUE   BOTH  CLEAN   WIFI    BLUE   
drone                                                                           
AIR          0.375  0.369  0.226  0.276    15000  15000  15000  15000   15000   
DIS          0.257  0.274  0.145  0.166    10000  10000  10000  10000   10000   
INS          0.400  0.425  0.452  0.402    15000  15000  15000  15000   15000   
MIN          0.311  0.351  0.229  0.272    15000  15000  15000  15000   15000   
MP1          0.434  0.230  0.370  0.111    15000  15000  15000  15000   15000   
MP2          0.388  0.515  0.335  0.424    15000  15000  15000  15000   15000   
PHA          0.424  0.359  0.422  0.341    10000  15000  10000  15000   10000   

                             
subset   BOTH  CLEAN   WIFI  
drone                        
AIR     15000  15000  15000  
DIS     10000  10000  10000  
INS     15000  15000  15000  
MIN     15000  15000  15000  
MP1     15000  15000  15000  
MP2     15000  15000  15000  
PHA     15000  10000  15000